This script takes a pre-optimized ensemble configuration (from a JSON file)
and applies it to a hold-out test set to get final performance metrics.
It is the final step after running the Optuna weight optimization script.

This version has been refactored to include bootstrap resampling to calculate
95% confidence intervals for all relevant performance metrics.

1. Set Up Environment

In [ ]:
# --- 1. Imports ---
print("Importing libraries...")
# (Same imports as your original script)
!pip install albumentations torchinfo
!pip install git+https://github.com/qubvel/segmentation_models.pytorch

# --- 1. Imports ---

In [ ]:
# --- 1. Imports ---
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image
import gc
import time
from tqdm import tqdm
import torchvision.models as models
from datetime import datetime
import torch.nn.functional as F
from google.colab import userdata
import random
import seaborn as sns
import shutil
import smtplib
from email.mime.text import MIMEText
from torch.cuda.amp import autocast
import cv2
import segmentation_models_pytorch as smp
import timm
from torchinfo import summary
import pandas as pd
import json
from sklearn.metrics import auc as sklearn_auc
import warnings
import albumentations as A
from albumentations.pytorch import ToTensorV2
import zipfile
import re
from collections import defaultdict # <-- Make sure this is imported

In [ ]:
# --- 5. Configuration & Setup ---
print("Configuring environment...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ### REFACTORED: Simplified Configuration ###
# This is the only path you need to set. Point it to the output of the Optuna script.
ENSEMBLE_META_PATH = "/content/drive/MyDrive/RESULTS_REPORT_ENSEMBLE/CHILE/NOT_NORMALIZED/ENSEMBLE_OPTIMIZATION_03_11_2025_13_30_50.json" # <--- UPDATE THIS PATH

BATCH_SIZE = 32
WORKERS = 2
SEED = 42

# --- Dataset Source Path (for the TEST set) ---
DATASET_ZIP_DIR = '/content/drive/MyDrive/IA_MEDICA_SAMPLES/CHILE/NOT_NORMALIZED_CLEAN'
base_data_dir = '/content/dataset'

# --- Paths to the HOLD-OUT TEST SET ---
test_cancer_image_dir = os.path.join(base_data_dir, 'TEST/CANCER')
test_cancer_mask_dir = os.path.join(base_data_dir, 'TEST/CANCER_MASK')
test_not_cancer_image_dir = os.path.join(base_data_dir, 'TEST/NOT_CANCER')
test_not_cancer_mask_dir = os.path.join(base_data_dir, 'TEST/NOT_CANCER_MASK')

DECODER_DROPOUT = 0

In [ ]:
# --- Seeding ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Configuration complete.")

In [ ]:
# --- Helper Functions ---
def get_formatted_datetime_string():
  now = datetime.now()
  return now.strftime("%d_%m_%Y_%H_%M_%S")

In [ ]:
def get_model(architecture, encoder,validation=False):

  if validation:
    aux_params=None
  else:
    aux_params=dict(dropout=DECODER_DROPOUT, classes=2)

  encoder_weights = None if validation else "imagenet"

  if architecture=="SWIN":
    model = smp.Unet(
    encoder_name=encoder,
    encoder_weights=encoder_weights,
    in_channels=3,
    classes=2,
    activation=None,
    decoder_attention_type=None,
    aux_params=aux_params)
  elif architecture=="DEEPLABV3PLUS":
    model = smp.DeepLabV3Plus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="INCEPTIONRESNETV2":
    model = smp.Unet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="DPT":
    model = smp.DPT(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        decoder_readout='ignore',
        aux_params=aux_params)
  elif architecture=="UNET++":
    model = smp.UnetPlusPlus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="FPN":
    model = smp.FPN(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="SEGFORMER":
    model = smp.Segformer(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="MANET":
    model = smp.MAnet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="UPERNET": # <-- ADD THIS BLOCK
    model = smp.UPerNet(
        encoder_name=encoder,
        encoder_weights="imagenet",
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  else:
    raise ValueError(f"Unknown architecture: {architecture}")

  return model

4. Create Custom Dataset

In [ ]:
# --- 8. Dataset Class (Simplified) ---
print("Defining simplified ProstateCancerDataset...")
# Use albumentations for basic transforms (Resize, Normalize, ToTensor)
class ProstateCancerDataset(Dataset):
    def __init__(self, cancer_image_dir, cancer_mask_dir, not_cancer_image_dir, not_cancer_mask_dir):
        # Removed is_train flag as augmentations are pre-applied
        self.cancer_image_dir = cancer_image_dir
        self.cancer_mask_dir = cancer_mask_dir
        self.not_cancer_image_dir = not_cancer_image_dir
        self.not_cancer_mask_dir = not_cancer_mask_dir

        # --- Base Transformation (Applied to ALL data) ---
        self.base_transform = A.Compose([
            A.Resize(224, 224, interpolation=cv2.INTER_LINEAR), # Specify interpolation
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(), # Handles image normalization (scaling) and channel order (C, H, W)
                          # Converts mask to Tensor (C, H, W)
        ])

        # --- Load File Lists ---
        # Defensive listing: check if dirs exist
        self.cancer_images = []
        if os.path.isdir(self.cancer_image_dir):
             self.cancer_images = [f for f in os.listdir(cancer_image_dir) if f.lower().endswith('.png')]
        else: print(f"Warning: Directory not found: {self.cancer_image_dir}")

        self.not_cancer_images = []
        if os.path.isdir(self.not_cancer_image_dir):
             self.not_cancer_images = [f for f in os.listdir(not_cancer_image_dir) if f.lower().endswith('.png')]
        else: print(f"Warning: Directory not found: {self.not_cancer_image_dir}")


        # Combine and store file paths and labels
        self.image_paths = []
        self.mask_paths = []
        self.labels = [] # Still useful maybe for checks later

        # --- MODIFICATION: Add a list to store patient IDs ---
        self.patient_ids = []

        # --- MODIFICATION: Compile the regex for efficiency ---
        patient_id_pattern = re.compile(r'PATIENT_(\d+)_')

        for img_name in self.cancer_images:
             img_path = os.path.join(self.cancer_image_dir, img_name)
             mask_path = os.path.join(self.cancer_mask_dir, img_name)

             # --- MODIFICATION: Extract patient ID ---
             match = patient_id_pattern.search(img_name)
             if os.path.isfile(mask_path) and match: # Ensure both mask and ID exist
                 self.image_paths.append(img_path)
                 self.mask_paths.append(mask_path)
                 self.labels.append(1)
                 self.patient_ids.append(match.group(1)) # Store the patient ID
             else: print(f"Warning: Mask or Patient ID missing for cancer image {img_name}")

        for img_name in self.not_cancer_images:
             img_path = os.path.join(self.not_cancer_image_dir, img_name)
             mask_path = os.path.join(self.not_cancer_mask_dir, img_name)

             # --- MODIFICATION: Extract patient ID ---
             match = patient_id_pattern.search(img_name)
             if os.path.isfile(mask_path) and match: # Ensure both mask and ID exist
                 self.image_paths.append(img_path)
                 self.mask_paths.append(mask_path)
                 self.labels.append(0)
                 self.patient_ids.append(match.group(1)) # Store the patient ID
             else: print(f"Warning: Mask or Patient ID missing for non-cancer image {img_name}")

    def __len__(self):
        # Length is simply the total number of valid image/mask pairs found
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        # --- MODIFICATION: Get the patient ID for this index ---
        patient_id = self.patient_ids[idx]

        try:
            # Load image using OpenCV (as Albumentations often uses it) - loads BGR
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None: raise IOError("cv2.imread failed for image")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert to RGB for consistency if needed downstream

            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None: raise IOError("cv2.imread failed for mask")

        except Exception as e:
            print(f"Error loading image/mask: {img_path} / {mask_path} - {e}")
            # Return None tuple, handled by collate_fn
            return None, None, None

        # Create Two-Channel Mask (One-Hot Encode) before transform
        mask = mask.astype(np.uint8)
        two_channel_mask = np.zeros((mask.shape[0], mask.shape[1], 2), dtype=np.float32)
        two_channel_mask[mask == 0, 0] = 1.0  # Background channel
        two_channel_mask[mask != 0, 1] = 1.0  # Cancer channel

        # Apply the base transformations
        try:
            # Pass mask correctly (shape H, W, C)
            augmented = self.base_transform(image=image, mask=two_channel_mask)
            final_image = augmented['image'] # Shape (C, H, W), FloatTensor, Normalized
            final_mask = augmented['mask']   # Shape (C, H, W), FloatTensor, Values 0.0 or 1.0

            # Ensure mask shape is (2, 224, 224)
            if final_mask.shape[0] != 2:
                 resized_mask = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)
                 two_channel_mask_resized = np.zeros((224, 224, 2), dtype=np.float32)
                 two_channel_mask_resized[resized_mask == 0, 0] = 1.0
                 two_channel_mask_resized[resized_mask != 0, 1] = 1.0
                 final_mask = torch.from_numpy(two_channel_mask_resized).permute(2, 0, 1) # HWC -> CHW

            # Final check on mask shape
            if final_mask.shape != (2, 224, 224):
                 raise ValueError(f"Final mask shape is incorrect: {final_mask.shape}")


        except Exception as e:
             print(f"Error applying transform to {os.path.basename(img_path)}: {e}")
             return None, None, None # Return None tuple on transform error


        return final_image, final_mask, patient_id

print("Dataset definition complete.")

In [ ]:
# --- 11. Utility Functions (Keep GPU Clear, Error Analysis, Email, Visualize) ---
print("Defining utility functions...")
def clear_gpu():
    # ... (clear_gpu remains the same) ...
    if torch.cuda.is_available():
      print("Clearing GPU cache...");
      torch.cuda.empty_cache();
      gc.collect();
      print("GPU cache cleared.");
      time.sleep(2)

In [ ]:
# Helper function for correct metric calculation (Action Point 4)
def calculate_metrics(tp, fp, fn, tn):
    # Dice and IoU for empty masks (true negatives) are 1.0
    is_true_negative = (tp + fp + fn) == 0
    dice = 1.0 if is_true_negative else (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
    iou = 1.0 if is_true_negative else tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    return {
        'dice': dice, 'iou': iou,
        'accuracy': (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0,
        'tpr': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        'tnr': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'precision': tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        'fnr': fn / (fn + tp) if (fn + tp) > 0 else 0.0
    }

In [ ]:
# Helper for printing
def print_metric_line(metric, pe, ci):
    ci_str = f"(95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])" if not np.isnan(ci[0]) else "(CI not calculated; n_patients < 20)"
    print(f"  {metric.replace('_',' ').title():<12}: {pe:.4f}  {ci_str}")

In [ ]:
def analyze_ensemble_metrics(models_list, model_weights, test_loader, device, optimal_threshold, num_auc_steps=101):
    """
    Calculates performance metrics for the ensemble.

    ### FINAL & COMPLETE VERSION (with Patient-Level Bootstrap) ###
    This function calculates and reports 95% Confidence Intervals for BOTH:
    1.  MICRO-AVERAGES (Overall pixel-level metrics) using a patient-level bootstrap.
    2.  MACRO-AVERAGES (Mean of per-image scores) using a score-level bootstrap.
    The AUC calculation is fully integrated.

    This version now correctly implements a patient-level (cluster) bootstrap.
    It includes a gatekeeper to only run the bootstrap if n_patients >= 20.
    """

    num_models = len(models_list)
    if not models_list:
      return None
    if model_weights is not None and len(model_weights) != num_models:
        raise ValueError("Length of model_weights must match the number of models.")
    if model_weights is None: # Use equal weights
        model_weights = [1.0 / num_models] * num_models
        print(f"Using equal weights for {num_models} models.")
    else: # Normalize provided weights
        sum_weights = sum(model_weights)
        if not np.isclose(sum_weights, 1.0):
          model_weights = [w/sum_weights for w in model_weights]; print(f"Normalized weights: {model_weights}")
        else:
          print(f"Using provided weights: {model_weights}")

    for model in models_list:
      model.eval()

    # --- 2. DATA COLLECTION LOOP (PATIENT-AWARE) ---
    stats_by_patient = defaultdict(list) # Groups patch stats [{tp, fp, fn, tn}, ...] by patient_id
    auc_thresholds = np.linspace(0.0, 1.0, num_auc_steps)
    auc_pixel_counts = np.zeros((num_auc_steps, 4), dtype=np.int64) # Stores [tp, fp, fn, tn] for each threshold

    print(f"Running Ensemble test evaluation with optimal threshold: {optimal_threshold:.4f}")

    with torch.no_grad():
        pbar = tqdm(test_loader, desc="Ensemble Test", leave=False)
        for batch_data in pbar:
            if batch_data is None or batch_data[0] is None:
              continue
            images, masks, patient_ids = batch_data
            images=images.to(device, non_blocking=True)
            true_indices = torch.argmax(masks, dim=1).int()
            current_batch_size = images.size(0)
            if current_batch_size == 0: continue

            # --- Ensemble Prediction ---
            sum_weighted_probs_cancer = torch.zeros_like(images[:, 0, :, :], device=device, dtype=torch.float32)
            valid_model_outputs = 0
            for i, model in enumerate(models_list):
                try:
                    with torch.amp.autocast('cuda'):
                        outputs_raw = model(images)
                        outputs = outputs_raw[0] if isinstance(outputs_raw, tuple) else outputs_raw
                        if outputs.shape[1] == 2:
                            probs_cancer = torch.softmax(outputs, dim=1)[:, 1, :, :]
                        else:
                            probs_cancer = torch.sigmoid(outputs).squeeze(1)
                        sum_weighted_probs_cancer += model_weights[i] * probs_cancer
                        valid_model_outputs += 1
                except Exception as model_err:
                    print(f"Warning: Model {i} failed on a batch: {model_err}. Skipping.")
                    continue
            if valid_model_outputs == 0:
                batches_skipped += 1
                continue
            ensemble_probs_cancer = sum_weighted_probs_cancer

            # --- Store patch stats grouped by patient ---
            ensemble_preds_opt = (ensemble_probs_cancer > optimal_threshold).int().cpu()
            true_indices_cpu = true_indices.cpu()

            # --- MODIFICATION: Group stats by patient ID ---
            for j in range(current_batch_size):
                pred_single, true_single = ensemble_preds_opt[j], true_indices_cpu[j]
                patient_id = patient_ids[j] # Get the patient ID for this image

                tp = ((pred_single == 1) & (true_single == 1)).sum().item()
                fp = ((pred_single == 1) & (true_single == 0)).sum().item()
                fn = ((pred_single == 0) & (true_single == 1)).sum().item()
                tn = ((pred_single == 0) & (true_single == 0)).sum().item()

                # Append the stats for this patch to the correct patient's list
                stats_by_patient[patient_id].append({'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn})

            # --- Accumulate counts for AUC curve (unchanged) ---
            ensemble_probs_cancer_cpu = ensemble_probs_cancer.cpu()
            for i, auc_thresh in enumerate(auc_thresholds):
                preds_binary_auc = (ensemble_probs_cancer_cpu > auc_thresh).int()
                auc_pixel_counts[i, 0] += ((preds_binary_auc == 1) & (true_indices_cpu == 1)).sum().item() # TP
                auc_pixel_counts[i, 1] += ((preds_binary_auc == 1) & (true_indices_cpu == 0)).sum().item() # FP
                auc_pixel_counts[i, 2] += ((preds_binary_auc == 0) & (true_indices_cpu == 1)).sum().item() # FN
                auc_pixel_counts[i, 3] += ((preds_binary_auc == 0) & (true_indices_cpu == 0)).sum().item() # TN

            del images, masks, true_indices, sum_weighted_probs_cancer, ensemble_preds_opt, true_indices_cpu, ensemble_probs_cancer_cpu
            torch.cuda.empty_cache()

    # --- Post-Loop Calculations ---
    if not stats_by_patient:
        print("Error: No patients were processed successfully.")
        return None

    # --- ACTION POINT 2.1: Count unique patients ---
    unique_patient_ids = list(stats_by_patient.keys())
    n_patients = len(unique_patient_ids)

    # --- ACTION POINT 2.2: Implement the Conditional Gatekeeper ---
    run_bootstrap = n_patients >= 20

    if not run_bootstrap:
        print("\n" + "="*80)
        print("WARNING: You cannot run bootstrap strategy with accuracy for a cluster with less than 20 patients minimum.")
        print("Script will only calculate the metrics with no confidence intervals.")
        print("="*80 + "\n")


    # Define metric keys and bootstrap samples (will only be used if run_bootstrap is True)
    n_bootstrap_samples = 10000
    metric_keys = ['dice', 'iou', 'accuracy', 'tpr', 'tnr', 'precision', 'fpr', 'fnr']

    # --- Initialize results dictionaries ---
    # These will be populated differently depending on the bootstrap condition.
    final_micro_metrics = {'point_estimate': {}, 'ci': {key: [np.nan, np.nan] for key in metric_keys}}
    final_macro_metrics = {'point_estimate': {}, 'ci': {key: [np.nan, np.nan] for key in metric_keys}}

    # --- 4. POINT ESTIMATE CALCULATIONS (RUN ALWAYS) ---
    print("\nCalculating Point Estimate Metrics from the full test set...")

    # Micro-Averages (from all patches)
    all_patches_stats = [stat for pat_stats in stats_by_patient.values() for stat in pat_stats]
    tp_micro, fp_micro, fn_micro, tn_micro = np.sum([list(s.values()) for s in all_patches_stats], axis=0)
    final_micro_metrics['point_estimate'] = calculate_metrics(tp_micro, fp_micro, fn_micro, tn_micro)

    # Macro-Averages (mean of per-patient scores)
    per_patient_scores = {key: np.zeros(n_patients) for key in metric_keys}
    for i, patient_id in enumerate(unique_patient_ids):
        tp_pat, fp_pat, fn_pat, tn_pat = np.sum([list(s.values()) for s in stats_by_patient[patient_id]], axis=0)
        patient_metrics = calculate_metrics(tp_pat, fp_pat, fn_pat, tn_pat)
        for key in metric_keys:
            per_patient_scores[key][i] = patient_metrics[key]

    for metric in metric_keys:
        final_macro_metrics['point_estimate'][metric] = np.mean(per_patient_scores[metric])

    # --- 5. CONDITIONAL BOOTSTRAP FOR CONFIDENCE INTERVALS ---
    if run_bootstrap:
        print(f"\nCalculating 95% CIs with Patient-Level Bootstrap ({n_bootstrap_samples} resamples)...")
        np.random.seed(SEED) # For reproducibility

        # Bootstrap for MICRO-AVERAGED Metrics
        bootstrap_micro_metrics = {key: np.zeros(n_bootstrap_samples) for key in metric_keys}
        for i in tqdm(range(n_bootstrap_samples), desc="Bootstrap (Micro)", leave=False):
            resampled_patient_ids = np.random.choice(unique_patient_ids, size=n_patients, replace=True)
            bootstrap_stats_sample = [stats_by_patient[pid] for pid in resampled_patient_ids]
            all_bootstrap_patches = [stat for pat_stats in bootstrap_stats_sample for stat in pat_stats]
            tp, fp, fn, tn = np.sum([list(s.values()) for s in all_bootstrap_patches], axis=0)
            metrics = calculate_metrics(tp, fp, fn, tn)
            for key in metric_keys:
                bootstrap_micro_metrics[key][i] = metrics[key]
        for metric in metric_keys:
            final_micro_metrics['ci'][metric] = np.percentile(bootstrap_micro_metrics[metric], [2.5, 97.5])

        # Bootstrap for MACRO-AVERAGED Metrics
        bootstrap_macro_means = {key: np.zeros(n_bootstrap_samples) for key in metric_keys}
        for i in tqdm(range(n_bootstrap_samples), desc="Bootstrap (Macro)", leave=False):
            for metric in metric_keys:
                 resampled_scores = np.random.choice(per_patient_scores[metric], size=n_patients, replace=True)
                 bootstrap_macro_means[metric][i] = np.mean(resampled_scores)
        for metric in metric_keys:
            final_macro_metrics['ci'][metric] = np.percentile(bootstrap_macro_means[metric], [2.5, 97.5])

    # --- 6. AUC CALCULATION ---
    auc_score = 0.0
    print("\nCalculating Ensemble AUC from accumulated counts...")
    try:
        total_pos = auc_pixel_counts[0, 0] + auc_pixel_counts[0, 2] # Initial TP + FN
        total_neg = auc_pixel_counts[0, 1] + auc_pixel_counts[0, 3] # Initial FP + TN
        if total_pos > 0 and total_neg > 0:
            tpr_values = auc_pixel_counts[:, 0] / total_pos
            fpr_values = auc_pixel_counts[:, 1] / total_neg
            auc_score = sklearn_auc(fpr_values, tpr_values)
    except Exception as e:
        print(f"AUC Calculation Error: {e}")


    # --- 7. FINAL REPORTING (FLEXIBLE) ---
    print("\n" + "="*25 + " Final Performance Summary " + "="*25)
    print("\n--- Overall Pixel-Level Metrics (Micro-Averages) ---")
    for metric in sorted(metric_keys):
        print_metric_line(metric, final_micro_metrics['point_estimate'][metric], final_micro_metrics['ci'][metric])
    print(f"  {'AUC':<12}: {auc_score:.4f}  (CI not calculated via bootstrap)")

    print("\n--- Mean Per-Patient Metrics (Macro-Averages) ---")
    for metric in sorted(metric_keys):
        print_metric_line(metric, final_macro_metrics['point_estimate'][metric], final_macro_metrics['ci'][metric])

    # --- 8. VISUALIZATION ---
    cm = np.array([[tn_micro, fp_micro], [fn_micro, tp_micro]])
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm / np.sum(cm), annot=True, fmt='.2%', cmap='Blues', xticklabels=['Pred NoCancer','Pred Cancer'], yticklabels=['True NoCancer','True Cancer'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Ensemble Confusion Matrix (Overall Pixel %)')
    plt.show()

    return {
        'micro_averaged_metrics': final_micro_metrics,
        'macro_averaged_metrics': final_macro_metrics,
        'auc': auc_score
    }

In [ ]:
def visualize_ensemble_predictions(models_list, model_weights, dataloader, device, threshold, num_samples=5):
    # ... (Similar structure to evaluate_ensemble, but plots images) ...
    num_models=len(models_list)
    if model_weights is None:
      model_weights = [1.0/num_models]*num_models
    for model in models_list:
      model.eval()
    print(f"Visualizing {num_samples} ENSEMBLE samples (Threshold={threshold:.4f})...")
    with torch.no_grad():
        try:
          batch_data = next(iter(dataloader))
        except StopIteration:
          print("DataLoader empty.");
          return
        if batch_data is None:
          print("Cannot load batch.");
          return
        images, masks, _ = batch_data;
        actual_batch_size = images.shape[0];
        num_samples = min(num_samples, actual_batch_size)
        if num_samples == 0:
          print("No samples.");
          return
        sample_indices = random.sample(range(actual_batch_size), num_samples)
        images_vis = images[sample_indices].to(device);
        masks_vis = masks[sample_indices] # Keep mask on CPU
        true_classes = torch.argmax(masks_vis, dim=1).numpy() # N, H, W

        # Get ensemble predictions
        sum_weighted_probs_cancer = torch.zeros_like(images_vis[:,0,:,:], device=device, dtype=torch.float32)
        for i, model in enumerate(models_list):
             with torch.amp.autocast('cuda'):
                  outputs_raw=model(images_vis);
                  if isinstance(outputs_raw, tuple):
                    outputs = outputs_raw[0]
                  else:
                    outputs = outputs_raw
                  if not isinstance(outputs, torch.Tensor):
                    continue
                  probs_cancer=torch.softmax(outputs,dim=1)[:,1,:,:]; sum_weighted_probs_cancer+=model_weights[i]*probs_cancer
        ensemble_probs_cancer = sum_weighted_probs_cancer # Already averaged if weights sum to 1
        ensemble_preds = (ensemble_probs_cancer > threshold).cpu().numpy().astype(np.uint8) # N, H, W

        images_np = images_vis.cpu().numpy() # N, C, H, W
        for j in range(num_samples):
            image = images_np[j].transpose((1,2,0));
            mean=np.array([0.485,0.456,0.406]);
            std=np.array([0.229,0.224,0.225]);
            image=std*image+mean;
            image=np.clip(image,0,1);
            pred_mask=ensemble_preds[j];
            true_mask=true_classes[j]
            fig,axes=plt.subplots(1,3,figsize=(15,5));
            axes[0].imshow(image);
            axes[0].set_title(f"Image {j+1}");
            axes[0].axis('off');
            axes[1].imshow(image);
            axes[1].imshow(np.ma.masked_where(pred_mask==0,pred_mask),cmap='jet',alpha=0.5);
            axes[1].set_title("Ensemble Pred Mask");
            axes[1].axis('off');
            axes[2].imshow(image);
            axes[2].imshow(np.ma.masked_where(true_mask==0,true_mask),cmap='jet',alpha=0.5);
            axes[2].set_title("True Mask");
            axes[2].axis('off');
            plt.tight_layout();
            plt.show()

print("Utility functions defined.")

In [ ]:
def print_scientific_analysis_report(metrics_results):
    """
    Prints a detailed, objective, and scientific guide for interpreting the model's
    performance, now including explanations for patient-level bootstrapping and CIs.
    """
    if not metrics_results:
        print("\nMetrics object is empty. Cannot generate report.")
        return

    # --- Unpack all the necessary data ---
    micro = metrics_results.get('micro_averaged_metrics', {})
    macro = metrics_results.get('macro_averaged_metrics', {})
    auc = metrics_results.get('auc')

    if not micro or not macro or auc is None:
        print("\nMetrics object is missing required data. Cannot generate full report.")
        return

    micro_pe = micro.get('point_estimate', {})
    micro_ci = micro.get('ci', {})
    macro_pe = macro.get('point_estimate', {})
    macro_ci = macro.get('ci', {})

    # --- Helper to format the CI string dynamically ---
    def get_ci_string(ci_data):
        if ci_data and not np.isnan(ci_data[0]):
            return f"(95% CI: [{ci_data[0]:.4f}, {ci_data[1]:.4f}])"
        else:
            return "(CI not calculated; n_patients < 20)"

    print("\n" + "="*30 + " Scientific Performance Analysis " + "="*30)
    print("\nThis report provides a scientific context for the model's performance on the hold-out test set.")
    print("It emphasizes patient-level generalization and statistical uncertainty.")

    print("\n\n" + "-"*25 + " Analysis of Key Metrics " + "-"*25)

    # --- 1. Segmentation Quality (Dice & IoU) ---
    print("\n[--- Segmentation Quality: Dice and IoU ---]")
    print("  - Definition: Measures the spatial overlap between predicted and true masks (Ideal = 1.0).")
    print("  - Micro-Average: Aggregates all pixels from all patients. This reflects overall pixel-level accuracy")
    print("    but can be dominated by patients who contributed a large number of patches.")
    print(f"    - Micro-Average Dice: {micro_pe.get('dice', 0):.4f} {get_ci_string(micro_ci.get('dice'))}")

    print("\n  - Macro-Average: Calculates the metric for each patient first, then averages these scores. This is the")
    print("    primary metric for clinical generalization, as it treats each patient equally.")
    print(f"    - Macro-Average Dice (Per-Patient Mean): {macro_pe.get('dice', 0):.4f} {get_ci_string(macro_ci.get('dice'))}")

    print("\n  - Interpretation of the 95% Confidence Interval (CI): The CI provides a plausible range for the true")
    print("    performance metric. A narrow CI suggests that the model's performance is stable and consistent")
    print("    across different patients in the test set.")

    # --- 2. Clinical Reliability: Sensitivity (TPR) and Miss Rate (FNR) ---
    print("\n[--- Clinical Reliability: Sensitivity / Miss Rate ---]")
    print("  - Definition (FNR): The False Negative Rate, or 'Miss Rate' (Ideal = 0.0). It is the proportion of")
    print("    cancerous regions/patients that the model failed to detect.")
    print("  - Interpretation: This metric is critical for clinical safety. A low FNR is essential for a screening tool.")
    print(f"  - The model's Macro-Average FNR is {macro_pe.get('fnr', 0):.4f} {get_ci_string(macro_ci.get('fnr'))}. This suggests that, on average,")
    print(f"    the model is expected to miss approximately {macro_pe.get('fnr', 0):.2%} of cancerous patients/regions.")

    # --- 3. Clinical Reliability: Specificity (TNR) and False Alarm Rate (FPR) ---
    print("\n[--- Clinical Reliability: Specificity / False Alarm Rate ---]")
    print("  - Definition (FPR): The False Positive Rate, or 'False Alarm Rate' (Ideal = 0.0). It is the proportion")
    print("    of healthy regions/patients that were incorrectly flagged as cancerous.")
    print("  - Interpretation: This metric is important for clinical efficiency, reducing unnecessary reviews.")
    print(f"  - The model's Macro-Average FPR is {macro_pe.get('fpr', 0):.4f} {get_ci_string(macro_ci.get('fpr'))}. This suggests that, on average,")
    print(f"    an estimated {macro_pe.get('fpr', 0):.2%} of non-cancerous patients/regions would trigger a false alarm.")

    # --- 4. Overall Discriminative Power (AUC) ---
    print("\n[--- Overall Discriminative Power: AUC ---]")
    print("  - Definition: The Area Under the ROC Curve measures the model's ability to distinguish between")
    print("    positive and negative pixels across all possible thresholds (Ideal = 1.0).")
    print(f"  - The model's pixel-level AUC is {auc:.4f}. As a benchmark, values above 0.95 typically reflect")
    print("    excellent discriminative power between the classes.")

    print("\n\n" + "="*25 + " How to Form a Conclusion " + "="*25)
    print("A robust and generalizable model demonstrates a combination of strengths:")
    print("  1. High Technical Skill: Indicated by a high Micro-Average Dice and a high AUC.")
    print("  2. High Generalization to New Patients: Indicated by a strong Macro-Average (per-patient) Dice score.")
    print("  3. High Safety & Sensitivity: Indicated by a low Macro-Average FNR.")
    print("  4. High Efficiency & Specificity: Indicated by a low Macro-Average FPR.")
    print("  5. High Confidence: Indicated by narrow 95% Confidence Intervals on the key macro-average metrics.")
    print("\nEvaluate these metrics based on the intended clinical application. For a screening tool, a low")
    print("FNR and its upper CI bound are paramount. For a confirmatory tool, a low FPR may be more critical.")
    print("="*80)

In [ ]:
# ==============================================================================
# --- Main Ensemble Evaluation ---
# ==============================================================================
print(f"\n{'='*25} Starting Final Ensemble Evaluation on Test Set {'='*25}")

# --- Extract Dataset ---
print("Extracting test dataset...")
fold_zip_filename = 'MASTER_SET_1.zip' # Assuming test data is in this zip
fold_zip_path = os.path.join(DATASET_ZIP_DIR, fold_zip_filename)
with zipfile.ZipFile(fold_zip_path, 'r') as z:
    z.extractall(base_data_dir)

In [ ]:
# --- Create Test DataLoader ---
print("\nCreating Test DataLoader...")
def collate_fn(batch):
    batch = list(filter(lambda x: x is not None and x[0] is not None and x[2] is not None, batch))
    if not batch:
      # Return a tuple of Nones that the main loop can check for
      return (None, None, None)

    return torch.utils.data.dataloader.default_collate(batch)

test_ds = ProstateCancerDataset(test_cancer_image_dir, test_cancer_mask_dir, test_not_cancer_image_dir, test_not_cancer_mask_dir)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True, collate_fn=collate_fn)
print(f"Test dataset size: {len(test_ds)}")

In [ ]:
# ### REFACTORED: Step 1 - Load the pre-computed ensemble recipe (NEW, IMPROVED VERSION) ###
print(f"\nLoading ensemble recipe from: {ENSEMBLE_META_PATH}")
if not os.path.exists(ENSEMBLE_META_PATH):
    raise FileNotFoundError(f"Ensemble metadata file not found! Path: {ENSEMBLE_META_PATH}")

with open(ENSEMBLE_META_PATH, 'r') as f:
    ensemble_recipe = json.load(f)

In [ ]:
# Extract the necessary information from the new format
optimal_threshold = ensemble_recipe['optimal_ensemble_threshold']
constituent_models_info = ensemble_recipe['constituent_models'] # This list now contains the weights

# Extract weights and model info in a single loop
optimal_weights = [model_info['ensemble_weight'] for model_info in constituent_models_info]

n_models = len(constituent_models_info)

In [ ]:
print(ensemble_recipe)

In [ ]:
# Extract the necessary information
#optimal_weights = ensemble_recipe['ensemble_weights']
optimal_threshold = ensemble_recipe['optimal_ensemble_threshold']
constituent_models_info = ensemble_recipe['constituent_models']
n_models = len(constituent_models_info)
print(f"Successfully loaded recipe for an ensemble of {n_models} models.")
print(f"Using Optimal Threshold: {optimal_threshold:.4f}")
print("Using Optimal Weights:")
for i, model_info in enumerate(constituent_models_info):
    arch = model_info.get('architecture', 'N/A')
    enc = model_info.get('encoder', 'N/A')
    weight = model_info.get('ensemble_weight', 'N/A')
    print(f"  - Weight: {weight:.4f} -> Model {i+1}: {arch} ({enc})")

In [ ]:
# ### REFACTORED: Step 2 - Load the EXACT models from the recipe ###
ensemble_models = []
print(f"\nLoading the {n_models} constituent models...")
for i, model_meta in enumerate(constituent_models_info):
    arch = model_meta.get('architecture')
    enc = model_meta.get('encoder')
    chkpt_path = model_meta.get('checkpoint_path')

    print(f" Loading Model {i+1}: {arch} ({enc}) from {os.path.basename(chkpt_path)}...")
    try:
        model_instance = get_model(architecture=arch, encoder=enc, validation=True)
        checkpoint = torch.load(chkpt_path, map_location=device)
        state_dict = checkpoint['model_state_dict']

        # # Handle compiled model state dict prefixes
        # if list(state_dict.keys())[0].startswith('_orig_mod.'):
        #       state_dict = {k.replace('_orig_mod.',''): v for k, v in state_dict.items()}

        if hasattr(model_instance, '_orig_mod') and not list(state_dict.keys())[0].startswith('_orig_mod.'):
              state_dict = {'_orig_mod.'+k: v for k, v in state_dict.items()} # Add prefix
        elif not hasattr(model_instance, '_orig_mod') and list(state_dict.keys())[0].startswith('_orig_mod.'):
              state_dict = {k.replace('_orig_mod.',''): v for k, v in state_dict.items()} # Remove prefix

        model_instance.load_state_dict(state_dict, strict=False) # Use strict=True for final loading
        model_instance.to(device)
        model_instance.eval()

        # Re-compile if the original was compiled for max performance
        # (Assuming 'is_compiled' would be in checkpoint, add if needed)
        # model_instance = torch.compile(model_instance)

        # --- Optional: Compile loaded model if original was compiled ---
        is_compiled = checkpoint.get('is_compiled', False)
        if is_compiled:
            print("  Compiling loaded model...")
            try:
              model_instance = torch.compile(model_instance)
            except Exception as e:
              print(f"   Compile failed: {e}. Using uncompiled.")

        ensemble_models.append(model_instance)
    except Exception as e:
        print(f"  FATAL ERROR loading model {os.path.basename(chkpt_path)}: {e}")
        # In a final run, a failure to load a required model should be a fatal error.
        exit(1)

if len(ensemble_models) != n_models:
    print("\nError: Number of loaded models does not match the recipe. Exiting.")
    exit(1)

print(f"\nSuccessfully loaded all {len(ensemble_models)} models.")
clear_gpu()

In [ ]:
# # ### REFACTORED: Step 3 - Directly evaluate on the Test Set ###
# The val_loader and find_optimal_ensemble_threshold call are REMOVED.
print("\n--- Evaluating Ensemble Performance on Hold-Out Test Set ---")
ensemble_metrics = None
try:
    ensemble_metrics = analyze_ensemble_metrics(
        models_list=ensemble_models,
        model_weights=optimal_weights,      # Use weights from the recipe
        test_loader=test_loader,
        device=device,
        optimal_threshold=optimal_threshold # Use threshold from the recipe
    )
except Exception as e:
    print(f"Error during final ensemble evaluation: {e}")
    import traceback; traceback.print_exc()

In [ ]:
# --- Visualization & Reporting (Adjusted for final return structure) ---
if ensemble_metrics:
    print("\n--- Visualizing Ensemble Predictions on Test Set ---")
    visualize_ensemble_predictions(
        models_list=ensemble_models,
        model_weights=optimal_weights,
        dataloader=test_loader,
        device=device,
        threshold=optimal_threshold,
        num_samples=5
    )

    ### REFACTORED: Final summary reporting from returned object ###
    # The analyze_ensemble_metrics function now prints its own detailed summary.
    # This section re-prints that summary from the returned object for final confirmation and logging.
    print("\n" + "="*20 + " Final Summary From Returned Object " + "="*20)

    # Extract the main results dictionaries
    micro_results = ensemble_metrics.get('micro_averaged_metrics')
    macro_results = ensemble_metrics.get('macro_averaged_metrics')
    auc_score = ensemble_metrics.get('auc')

    if not micro_results or not macro_results:
        print("Evaluation produced no valid metrics.")
    else:
        # --- Print the Micro-Averages with their CIs ---
        print("\n--- Overall Pixel-Level Metrics (Micro-Averages) ---")
        micro_pes = micro_results.get('point_estimate', {})
        micro_cis = micro_results.get('ci', {})

        for metric in sorted(micro_pes.keys()):
            pe = micro_pes[metric]
            ci = micro_cis.get(metric, [0.0, 0.0])
            print(f"  {metric.replace('_',' ').title():<12}: {pe:.4f}  (95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])")

        # Report the AUC score
        if auc_score is not None:
            print(f"  {'AUC':<12}: {auc_score:.4f}  (CI not calculated via bootstrap)")


        # --- Print the Macro-Averages with their CIs ---
        print("\n--- Mean Per-Image Metrics (Macro-Averages) ---")
        macro_pes = macro_results.get('point_estimate', {})
        macro_cis = macro_results.get('ci', {})

        for metric in sorted(macro_pes.keys()):
            pe = macro_pes[metric]
            ci = macro_cis.get(metric, [0.0, 0.0])
            print(f"  {metric.replace('_',' ').title():<12}: {pe:.4f}  (95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])")

else:
    print("\nEnsemble evaluation failed.")

# --- ADD THE CUSTOMIZED REPORT CALL HERE ---
if ensemble_metrics:
    print_scientific_analysis_report(ensemble_metrics)

print("\n--- Final Evaluation Script Finished ---")